# Health Checks: Verify Stack Readiness

## Overview

Before running tests, verify the entire stack is healthy. This notebook performs quick sanity checks on the control plane, wrapper services, and data layer.

**What you'll learn:**
- Check control plane health and readiness endpoints
- Create test resources (mapping, snapshot, instance) via SDK
- Verify wrapper connectivity and query execution
- Validate test user access

**Prerequisites:**
- Completed: [00_core_prerequisites.ipynb](./00_core_prerequisites.ipynb)
- Control plane must be running

**Time to complete:** 5 minutes

---

**Test Coverage:**
- Health endpoints (Control Plane)
- Basic connectivity
- Create mapping, snapshot, instance via SDK
- SDK imports work

**Execution:** Run this FIRST before other test notebooks.

In [ ]:
import os

# Parameters cell - papermill will inject values here
# Wrapper type for instance creation
WRAPPER_TYPE_STR = os.environ.get("WRAPPER_TYPE", "falkordb")  # or "ryugraph"

# Phase 2.3 Optimization: Support instance pooling
SKIP_RESOURCE_CREATION = False  # Set to True to use pool instance
POOL_INSTANCE_ID = None  # Instance ID from pool (when SKIP_RESOURCE_CREATION=True)

# Note: Auth is handled via TestPersona - env vars like GRAPH_OLAP_API_KEY_ANALYST_ALICE

## 1. SDK Import Test

Verify all SDK modules can be imported.

In [ ]:
import sys
import uuid
import os

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")
print(f"Using Bearer token: {'yes' if os.environ.get('GRAPH_OLAP_API_KEY') else 'no'}")

In [ ]:
# Test: Core SDK imports
from graph_olap import GraphOLAPClient
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

# Convert wrapper type string to enum
WRAPPER_TYPE = WrapperType(WRAPPER_TYPE_STR)

print("Smoke 1.1 PASSED: Core SDK imports successful")
print(f"  Wrapper type: {WRAPPER_TYPE.value}")

In [ ]:
# Test: Mapping model imports
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition

print("Smoke 1.2 PASSED: Mapping model imports successful")

## 2. Control Plane Health Checks

In [ ]:
from graph_olap import notebook

# Create test context with automatic cleanup
ctx = notebook.test("SmokeTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

print(f"Test context created for {client._config.api_url}")
print(f"  Persona: {TestPersona.ANALYST_ALICE.value.name}")

In [ ]:
# Test: Control Plane health endpoint
health = client.health.check()

assert health is not None, "Health response should not be None"
assert health.status == "healthy", f"Expected 'healthy', got '{health.status}'"

print(f"Smoke 2.1 PASSED: Control Plane health status='{health.status}'")

In [ ]:
# Test: Control Plane readiness endpoint
ready = client.health.ready()

assert ready is not None, "Ready response should not be None"
assert ready.status in ["ready", "healthy"], f"Expected ready status, got '{ready.status}'"

print(f"Smoke 2.2 PASSED: Control Plane readiness status='{ready.status}'")
if hasattr(ready, 'database') and ready.database:
    print(f"  Database: {ready.database}")

## 3. Create Test Data via SDK

This section creates a mapping, snapshot, and instance dynamically.
All test data is created via SDK (no database seeding).

In [ ]:
# Phase 2.3: Pool mode detection
if SKIP_RESOURCE_CREATION:
    # Using pool instance - no resource creation needed
    assert POOL_INSTANCE_ID is not None, "POOL_INSTANCE_ID required when SKIP_RESOURCE_CREATION=True"
    INSTANCE_ID = int(POOL_INSTANCE_ID)
    print(f"✓ Pool mode: Using pool instance {INSTANCE_ID} (skipping resource creation)")
else:
    # Standard mode - create resources dynamically
    TEST_RUN_ID = uuid.uuid4().hex[:8]
    MAPPING_NAME = f"SmokeTest-{TEST_RUN_ID}"
    print("✓ Standard mode: Creating resources dynamically")
    print(f"  Test run ID: {TEST_RUN_ID}")
    print(f"  Mapping name: {MAPPING_NAME}")

In [ ]:
# Cleanup is automatic with notebook.test()!
# Resources created via ctx.mapping(), ctx.snapshot(), ctx.instance() are auto-tracked
if not SKIP_RESOURCE_CREATION:
    print("Starting Smoke E2E Test - resources will be cleaned up automatically via atexit")

In [ ]:
# Define node and edge definitions (only in standard mode)
if not SKIP_RESOURCE_CREATION:
    # These reference the Trino source tables created by trino-seed job
    # Using SELECT DISTINCT because test data may have duplicates
    person_node = NodeDefinition(
        label="Person",
        sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
        primary_key={"name": "id", "type": "STRING"},
        properties=[
            PropertyDefinition(name="name", type="STRING"),
            PropertyDefinition(name="age", type="INT64"),
        ]
    )

    knows_edge = EdgeDefinition(
        type="KNOWS",
        from_node="Person",
        to_node="Person",
        sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
        from_key="from_id",
        to_key="to_id",
        properties=[
            PropertyDefinition(name="since", type="INT64"),
        ]
    )

    print(f"Node definition: {person_node.label} (primary_key: {person_node.primary_key})")
    print(f"Edge definition: {knows_edge.type} ({knows_edge.from_node} -> {knows_edge.to_node})")

In [ ]:
# Create mapping via ctx.mapping() (auto-tracked, only in standard mode)
if not SKIP_RESOURCE_CREATION:
    mapping = ctx.mapping(
        name=f"SmokeTest-{ctx.run_id}",
        description="Smoke test mapping - created dynamically via SDK",
        node_definitions=[person_node],
        edge_definitions=[knows_edge],
    )

    assert mapping is not None, "Mapping should not be None"
    assert mapping.id is not None, "Mapping should have an ID"

    MAPPING_ID = mapping.id
    print(f"Smoke 3.1 PASSED: Created mapping (id={MAPPING_ID}, name='{mapping.name}')")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED
# Create instance directly from mapping using create_from_mapping()
# The snapshot is created automatically in the background
if not SKIP_RESOURCE_CREATION:
    print(f"Creating instance directly from mapping (snapshot created automatically)...")
    print("  (This may take up to 3-4 minutes for snapshot export + instance startup)")

    instance = client.instances.create_from_mapping_and_wait(
        mapping_id=MAPPING_ID,
        name=f"SmokeTest-Instance-{ctx.run_id}",
        wrapper_type=WRAPPER_TYPE,
        timeout=300,  # Longer timeout for snapshot export
        poll_interval=5,
    )

    assert instance is not None, "Instance should not be None"
    assert instance.id is not None, "Instance should have an ID"
    assert instance.status == "running", f"Expected status 'running', got '{instance.status}'"

    INSTANCE_ID = instance.id
    SNAPSHOT_ID = instance.snapshot_id  # Get auto-created snapshot ID
    ctx._track('instance', INSTANCE_ID, instance.name)
    
    print(f"Smoke 3.2 PASSED: Created instance (id={INSTANCE_ID}, status='{instance.status}')")
    print(f"  Auto-created snapshot: (id={SNAPSHOT_ID})")
    if hasattr(instance, 'instance_url') and instance.instance_url:
        print(f"  Instance URL: {instance.instance_url}")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED - cell merged with cell-14
# Instance creation now happens via create_from_mapping() which handles both
# snapshot creation and instance creation in one step.
if not SKIP_RESOURCE_CREATION:
    print("Smoke 3.3 PASSED: Instance already created via create_from_mapping()")
    print(f"  Instance ID: {INSTANCE_ID}")
    print(f"  Snapshot ID: {SNAPSHOT_ID}")

## 4. Wrapper Health Checks

In [ ]:
# Connect to wrapper via SDK (environment-aware)
conn = client.instances.connect(INSTANCE_ID)
print(f"Connected to instance {INSTANCE_ID}")

In [ ]:
# Test: Wrapper can execute queries (readiness)
result = conn.query("MATCH (n) RETURN count(n) AS count LIMIT 1")

assert result is not None, "Query should succeed if wrapper is ready"
assert result.row_count == 1, "Should get 1 row"

node_count = result.rows[0][0]
print(f"Smoke 4.2 PASSED: Wrapper query works (node count={node_count})")

In [ ]:
# Test: NetworkX extension server available
try:
    algos = conn.networkx.algorithms()
    assert len(algos) > 0, "Should have NetworkX algorithms available"
    print(f"Smoke 4.3 PASSED: NetworkX extension available ({len(algos)} algorithms)")
except Exception as e:
    print(f"Smoke 4.3 WARNING: NetworkX extension not available - {e}")

## 5. Test Data Verification

In [ ]:
# Test: Mapping was created correctly (only in standard mode)
if not SKIP_RESOURCE_CREATION:
    fetched_mapping = client.mappings.get(MAPPING_ID)

    assert fetched_mapping is not None, f"Mapping {MAPPING_ID} should exist"
    assert fetched_mapping.id == MAPPING_ID

    print(f"Smoke 5.1 PASSED: Mapping exists (id={fetched_mapping.id}, name='{fetched_mapping.name}')")
else:
    print("Smoke 5.1 SKIPPED: Mapping verification (pool mode)")

In [ ]:
# Test: Snapshot is ready (only in standard mode)
if not SKIP_RESOURCE_CREATION:
    fetched_snapshot = client.snapshots.get(SNAPSHOT_ID)

    assert fetched_snapshot is not None, f"Snapshot {SNAPSHOT_ID} should exist"
    assert fetched_snapshot.id == SNAPSHOT_ID
    assert fetched_snapshot.status == "ready", f"Snapshot should be 'ready', got '{fetched_snapshot.status}'"

    print(f"Smoke 5.2 PASSED: Snapshot exists (id={fetched_snapshot.id}, status='{fetched_snapshot.status}')")
else:
    print("Smoke 5.2 SKIPPED: Snapshot verification (pool mode)")

In [ ]:
# Test: Instance is running
fetched_instance = client.instances.get(INSTANCE_ID)

assert fetched_instance is not None, f"Instance {INSTANCE_ID} should exist"
assert fetched_instance.id == INSTANCE_ID
assert fetched_instance.status == "running", f"Instance should be 'running', got '{fetched_instance.status}'"

print(f"Smoke 5.3 PASSED: Instance exists (id={fetched_instance.id}, status='{fetched_instance.status}')")

In [ ]:
# Test: Graph has expected data
person_count = conn.query_scalar("MATCH (p:Person) RETURN count(p)")
edge_count = conn.query_scalar("MATCH ()-[r:KNOWS]->() RETURN count(r)")

assert person_count >= 1, f"Should have at least 1 Person node, got {person_count}"
assert edge_count >= 1, f"Should have at least 1 KNOWS edge, got {edge_count}"

print(f"Smoke 5.4 PASSED: Graph has data ({person_count} Person nodes, {edge_count} KNOWS edges)")

## 6. Test Users Verification

In [ ]:
# Test: Can access API with different personas
# Test personas are accessed via ctx.with_persona()
from graph_olap.testing import TestPersona

# Test with different personas
test_personas = [
    TestPersona.ANALYST_ALICE,
    TestPersona.ANALYST_BOB,
    TestPersona.ADMIN_CAROL,
    TestPersona.OPS_DAVE,
]

for persona in test_personas:
    try:
        persona_client = ctx.with_persona(persona)
        mappings = persona_client.mappings.list()
        assert mappings is not None, f"{persona.value.name} should be able to list mappings"
        print(f"  ✓ {persona.value.name}: {len(mappings)} mappings visible")
    except ValueError as e:
        # API key not configured for this persona - OK in some environments
        print(f"  - {persona.value.name}: Not configured ({e})")

print(f"Smoke 6.1 PASSED: Test persona access verified")

## Summary

In [ ]:
# Cleanup is automatic via atexit!
# Resources created via ctx.mapping(), ctx.snapshot(), ctx.instance() will be cleaned up
if not SKIP_RESOURCE_CREATION:
    # Explicitly cleanup (optional - also happens on exit)
    results = ctx.cleanup()
    print(f"Cleanup: {results}")
else:
    print("Pool mode - no cleanup needed (pool manages instances)")

In [ ]:
# Close client
client.close()

import os

print("\n" + "="*60)
print("SMOKE TESTS PASSED - Stack is ready for full E2E tests")
print("="*60)
print("\nVerified:")
print("  1. SDK Imports: Core modules, models, exceptions")
print("  2. Control Plane: Health and readiness endpoints")

if SKIP_RESOURCE_CREATION:
    print(f"  3. Test Data: Used pool instance {INSTANCE_ID}")
else:
    print("  3. Test Data Creation:")
    print(f"     - Mapping ID: {MAPPING_ID}")
    print(f"     - Snapshot ID: {SNAPSHOT_ID}")
    print(f"     - Instance ID: {INSTANCE_ID}")

print("  4. Wrapper: Status, query execution, NetworkX extension")

if SKIP_RESOURCE_CREATION:
    print("  5. Data Verification: Instance, graph data (pool mode)")
else:
    print("  5. Data Verification: Mapping, snapshot, instance, graph data")

if os.environ.get("GRAPH_OLAP_API_KEY"):
    print("  6. Test Users: Bearer token identity verified")
else:
    print("  6. Test Users: All 4 test users can access API")

if SKIP_RESOURCE_CREATION:
    print(f"\nMode: POOL (instance {INSTANCE_ID})")
else:
    print(f"\nMode: STANDARD (test run ID: {TEST_RUN_ID})")